# 파싱
## 최근 7일 메일 가져오기

In [5]:
import imaplib
from datetime import datetime, timedelta


mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL, PASSWORD)
mail.select("INBOX")

# 최근 7일
since_date = (datetime.now() - timedelta(days=7)).strftime("%d-%b-%Y")

# 어제 날짜 00시 기준 : IMAP SINCE는 시간 단위 지원이 안되기 때문에 SINCE 어제날짜 하면 어제 00시 이후가 자동으로 됨
# since_date = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y")

_, msg_nums = mail.search(None, f'SINCE {since_date}')

num_list = msg_nums[0].split()
print(f"최근 7일 메일 수: {len(num_list)}건")

최근 7일 메일 수: 3건


## 제목/발신자/날짜 파싱

In [6]:
import email
from email.header import decode_header

def decode_str(s):
    if s is None:
        return ""
    result = ""
    for part, enc in decode_header(s):
        if isinstance(part, bytes):
            result += part.decode(enc or "utf-8", errors="ignore")
        else:
            result += str(part)
    return result

emails = []
for num in num_list[::-1]:  # 최신순
    _, data = mail.fetch(num, "(RFC822)")
    msg = email.message_from_bytes(data[0][1])

    emails.append({
        "subject": decode_str(msg["Subject"]),
        "sender":  decode_str(msg["From"]),
        "date":    msg.get("Date", ""),
    })

print(f" {len(emails)}건 파싱 완료")

 3건 파싱 완료


## 가져온 메일 파싱 결과 

In [7]:
import pandas as pd

df = pd.DataFrame(emails)
df

,subject,sender,date
0,Re: [업무협조] 데이터 검증 요청,민석 <mhnkms8041@gmail.com>,"Tue, 28 Apr 2026 11:42:36 +0900"
1,[업무협조] 데이터 검증 요청,윤여옥 <dudhr17050@naver.com>,"Tue, 28 Apr 2026 11:22:52 +0900"
2,[업무협조] 기능 검토 요청,윤여옥 <dudhr17050@naver.com>,"Tue, 28 Apr 2026 11:04:11 +0900"


## 접속 종료

In [ ]:
mail.logout()
print("접속 종료")

# PDF 다운로드